# TP2 : Classification — implémentation, évaluation et comparaison de méthodes

### Identification de l'étudiant

In [ ]:
# Nom : 
# Prénom : 
# Numéro d'étudiant : 

### Modalités de rendu

1. Chaque étudiant doit rendre un travail individuel.
2. Renommez ce fichier selon la convention : `TP2_Nom_Prenom.ipynb`.
3. Le rendu s'effectuera via le lien de dépôt communiqué par votre enseignant.
4. Assurez-vous que votre code s'exécute sans erreur (Menu : Kernel > Restart & Run All).

### Objectifs de la séance

Ce TP a pour objectif l’implémentation et la comparaison de méthodes de classification. Nous étudierons en particulier deux algorithmes classiques d’apprentissage supervisé :
- la régression logistique,
- l’algorithme des k plus proches voisins (k-NN).

Les performances des modèles seront évaluées et comparées à l’aide de métriques standards, notamment :
- la matrice de confusion,
- la précision (precision) et rappel (recall).
- ROC, AUC

### Documentation utile
- [Numpy User Guide](https://numpy.org/doc/stable/user/index.html)
- [Scikit-Learn OpenML](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html)


## Organisation du TP

Ce TP contient 5 parties : 4 obligatoires + 1 bonus :

1. **Régression Logistique Binaire implémentée à la main**
2. **Régression Logistique Multiclasse (Softmax) implémentée à la main**
3. **Utilisation des modèles scikit-learn et comparaison (Régression Logistique vs k-NN)**
4. **Jeu de données Moons : limites de la régression logistique et avantage du k-NN**
5. **Jeu de données Cancer du Sein (bonus): du modèle à 2 variables à l'évaluation clinique complète**



# Partie 1 — Régression Logistique Binaire (Implémentée à la Main)

Dans cette section, nous implémentons la **régression logistique from scratch** avec la descente de gradient.

Le modèle logistique estime la probabilité :

$$
P(Y=1|x)=\sigma(w^Tx+b)
$$

où la **fonction sigmoïde** est

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

Pour entraîner le modèle, on minimise la **perte d'entropie croisée binaire** :

$$
L = - \frac{1}{N} \sum_{i=1}^N
[y_i \log(p_i) + (1-y_i)\log(1-p_i)]
$$

Le gradient de cette perte par rapport à $w$ est :

$$
\nabla_w L = \frac{1}{N} X^T (p - y)
$$

> 💡 Assurez-vous de comprendre d'où vient cette formule de gradient avant de la coder.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Générer un jeu de données binaire simple

In [ ]:
# Cette cellule est fournie — exécutez-la simplement
np.random.seed(0)

N = 200
X = np.random.randn(N, 2)
true_w = np.array([2, -1])
logits = X @ true_w
y = (logits > 0).astype(int)

## Fonction sigmoïde

In [ ]:
def sigmoid(z):
    #TODO : implémenter la fonction sigmoïde
    pass

## Perte d'entropie croisée binaire et gradient

In [ ]:
def binary_cross_entropy(X, y, w):

    #TODO : calculer la combinaison linéaire z = X @ w
    z = ...

    #TODO : appliquer sigmoid pour obtenir les probabilités prédites p
    p = ...

    #TODO : calculer la perte d'entropie croisée binaire (voir formule ci-dessus)
    loss = ...

    #TODO : calculer le gradient (voir formule ci-dessus)
    # 
    grad = ...

    return loss, grad

## Optimiseur par descente de gradient

In [ ]:
def gradient_descent(X, y, lr=0.1, epochs=200):

    # Initialiser les poids à zéro
    w = np.zeros(X.shape[1])
    losses = []

    for _ in range(epochs):

        #TODO : appeler binary_cross_entropy pour obtenir loss et grad
        loss, grad = ...

        #TODO : mettre à jour les poids avec la règle de descente de gradient
        w = ...

        losses.append(loss)

    return w, losses

In [ ]:
# Run gradient descent and plot the training loss — plotting is provided
w, losses = ...

plt.figure(figsize=(6, 3))
plt.plot(..., color='steelblue')
plt.title("Perte d'entraînement — régression logistique binaire")
plt.xlabel(...)
plt.ylabel(...)
plt.tight_layout()
plt.show()


---

# Partie 2 — Régression Logistique Multiclasse (Softmax) Implémentée à la Main

Pour **K classes**, la régression logistique utilise la **fonction softmax**.

$$
\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

Chaque classe reçoit une probabilité et la somme des probabilités vaut 1.

## Perte d'Entropie Croisée Multiclasse

$$
L = - \frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{K} y_{ik} \log(p_{ik})
$$

Le gradient par rapport à la matrice de poids $W$ est :

$$
\nabla_W L = \frac{1}{N} X^T (P - Y)
$$


In [ ]:
def softmax(z):
    # Soustraire max(z) par ligne évite les débordements numériques — cette cellule est fournie
    exp = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp / exp.sum(axis=1, keepdims=True)

In [ ]:
def multiclass_cross_entropy(X, Y, W):

    # Indice : calculer logits = X @ W  (forme : N x K)
    logits = ...

    #TODO : appliquer softmax pour obtenir les probabilités de classe P
    P = ...

    #TODO : calculer la perte d'entropie croisée multiclasse
    # Indice : sommer sur les classes d'abord, puis moyenner sur les échantillons
    loss = ...

    # Indice : (même formule que le cas binaire, maintenant en matrice)
    grad = X.T @ (P-Y) / len(X)

    return loss, grad

## Générer un jeu de données multiclasse jouet

In [ ]:
# Cette cellule est fournie — exécutez-la simplement
np.random.seed(1)

N, D, K = 300, 2, 3
X = np.random.randn(N, D)
W_true = np.random.randn(D, K)
logits = X @ W_true
y = np.argmax(logits, axis=1)

# One-hot encode y into Y  (shape: N x K)
Y = np.eye(K)[y]

## Entraîner avec la descente de gradient

In [ ]:
def train_softmax(X, Y, lr=0.1, epochs=300):

    # Initialiser la matrice de poids W à zéro  (forme : D x K)
    W = np.zeros((X.shape[1], Y.shape[1]))
    losses = []

    for _ in range(epochs):

        #TODO : appeler multiclass_cross_entropy pour obtenir loss et grad
        loss, grad = ...

        #TODO : mettre à jour W
        W -= ...

        losses.append(loss)

    return W, losses

In [ ]:
# Entraînement et tracé — le tracé est fourni
W, losses = train_softmax(X, Y)

plt.figure(figsize=(6, 3))
plt.plot(losses, color='tomato')
plt.title("Perte d'entraînement — softmax multiclasse")
plt.xlabel("Époque")
plt.ylabel("Perte")
plt.tight_layout()
plt.show()


---

# Partie 3 — Utilisation des Modèles Scikit-Learn

Nous appliquons maintenant la **Régression Logistique** et les **k-Plus Proches Voisins** à des jeux de données réels.


In [ ]:
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

## Jeu de données Titanic (Classification Binaire)

Nous prédisons la survie sur le Titanic — une tâche binaire.

> 💡 On ne garde ici que les variables numériques et on supprime les lignes avec des valeurs manquantes.


In [ ]:
# Chargement et préparation du jeu Titanic — fourni
titanic = sns.load_dataset("titanic")

features = ['pclass', 'age', 'fare', 'sibsp', 'parch']
df = titanic[features + ['survived']].dropna()

X = df[features]
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
#TODO : entraîner une LogisticRegression (max_iter=1000) et afficher sa matrice de confusion
# Indice : .fit(), .predict(), confusion_matrix()


In [ ]:
#TODO : entraîner un KNeighborsClassifier (n_neighbors=5) et afficher sa matrice de confusion


### 📝 Observations

*Commentez les matrices de confusion. Quel modèle fait plus de faux positifs / faux négatifs ?  
Un modèle semble-t-il mieux adapté à cette tâche ?*



## Jeu de données Penguins (Classification Multiclasse)

Nous prédisons maintenant l'espèce de pingouin à partir de mesures morphologiques — tâche multiclasse.


In [ ]:
# Chargement et préparation du jeu Penguins — fourni
penguins = sns.load_dataset("penguins").dropna()

X = penguins[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]
y = penguins['species']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
#TODO : entraîner LogisticRegression (multinomial, max_iter=1000)
# Afficher la matrice de confusion et la précision


In [ ]:
#TODO : entraîner KNeighborsClassifier (n_neighbors=5)
# Afficher la matrice de confusion et la précision


### 📝 Observations

*Commentez les résultats. Quelle espèce est le plus souvent confondue ? Pourquoi ?*



## Comparaison entre notre Softmax fait à la main et Sklearn on Penguins

Nous appliquons maintenant le softmax construit en Partie 2 au jeu Penguins
et comparons ses performances avec sklearn.

> 💡 Notre implémentation attend :
> - `X` comme **tableau numpy** (pas un DataFrame)
> - `y` comme **labels entiers** (pas des chaînes)
> - `Y` comme **matrice one-hot**


In [ ]:
# Étape 1 : encoder les labels d'espèce en entiers 
# TODO : utiliser pd.get_dummies() sur penguins['species'] pour obtenir un encodage one-hot,
# puis convertir le résultat en tableau numpy. 
# Vous pouvez aussi utiliser: sklearn.preprocessing.LabelEncoder 

y_raw = ...

#TODO : extraire les 4 colonnes de variables en tableau numpy X_raw
X_raw = ...

print("Classes :", ...)
print("Exemple y_raw :", y_raw[:5])

In [ ]:
# Étape 2 : normaliser et diviser — fourni
sc = StandardScaler()
X_scaled = sc.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_raw, test_size=0.2, random_state=0)

#TODO : encoder y_tr en one-hot dans Y_tr  (forme : N x K)
# Indice : K = len(np.unique(y_raw)), puis np.eye(K)[y_tr]
K = ...
Y_tr = ...

In [ ]:
# Étape 3 : entraîner notre softmax fait à la main sur Penguins
#TODO : appeler train_softmax avec lr et epochs appropriés
W_penguins, losses_penguins = ...

#TODO : prédire sur X_te — calculer les logits puis prendre l'argmax
y_pred_hand = ...

acc_hand = np.mean(y_pred_hand == y_te)

# Référence sklearn pour comparaison
log_sk = LogisticRegression(max_iter=1000, multi_class="multinomial")
log_sk.fit(X_tr, y_tr)
acc_sklearn = log_sk.score(X_te, y_te)

print(f"Précision Softmax à la main       : {acc_hand:.3f}")
print(f"Précision Régression Logistique Sklearn : {acc_sklearn:.3f}")

In [ ]:
# Tracés de comparaison 
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Training loss
axes[0].plot(losses_penguins, color='steelblue')
axes[0].set_title("Softmax à la main — perte d'entraînement")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

species_names =  # TODO

# Confusion matrices
for ax, preds, title in zip(
    axes[1:],
    [y_pred_hand, log_sk.predict(X_te)],
    ["Softmax à la main", "Régression Logistique Sklearn"]
):
    import seaborn as sns
    cm = confusion_matrix(y_te, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=species_names, yticklabels=species_names)
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
    ax.set_title(title)

plt.tight_layout()
plt.show()

### 📝 Observations

*Comment votre implémentation se compare-t-elle à sklearn ?  
Qu'est-ce qui pourrait expliquer un écart de performance ?*




---


# Partie 4 - Données Non Linéairement Séparables (Jeu Moons)

La régression logistique suppose une **frontière de décision linéaire**.
Quand les classes ne sont pas linéairement séparables, elle ne peut pas bien s'adapter aux données.
k-NN ne fait **pas cette hypothèse** et peut capturer des frontières courbes et complexes.

Nous utilisons `make_moons` — deux demi-cercles imbriqués — comme illustration.


In [ ]:
# Générer le jeu moons — fourni
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=0)

# Visualiser les données brutes — given
plt.figure(figsize=(6, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='bwr', edgecolors='k', s=30)
plt.title("Jeu Moons — non linéairement séparable")
plt.tight_layout()
plt.show()

In [ ]:
#TODO : diviser X_moons / y_moons en train/test (test_size=0.2, random_state=0)
X_train_m, X_test_m, y_train_m, y_test_m = ...

#TODO : entraîner une LogisticRegression et un KNeighborsClassifier (k=5)
log_moons = ...
knn_moons = ...

# Entraîner les deux modèles
...

In [ ]:
# Tracés des frontières de décision — fournis, exécutez simplement cette cellule
from sklearn.inspection import DecisionBoundaryDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model, name in zip(
    axes,
    [log_moons, knn_moons],
    ["Régression Logistique", "k-NN (k=5)"]
):
    DecisionBoundaryDisplay.from_estimator(
        model, X_moons, cmap="bwr", alpha=0.3, ax=ax
    )
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons,
               cmap="bwr", edgecolors='k', s=30)
    acc = model.score(X_test_m, y_test_m)
    ax.set_title(f"{name}\nPrécision test : {acc:.2f}")

plt.tight_layout()
plt.show()

In [ ]:
#TODO : afficher la précision des deux modèles
print(f"Précision Régression Logistique : ...")
print(f"Précision k-NN                  : ...")

### 📝 Observations

*Regardez les deux frontières de décision. Quelle forme produit la régression logistique ?  
Pourquoi le k-NN fait-il mieux ici ?  
Pouvez-vous penser à des cas réels où les données seraient de même non linéairement séparables ?*




---

# Partie 5 — Cancer du Sein : Du Modèle à 2 Variables au Modèle Complet

Cette partie suit une **progression pédagogique** :

1. Identifier les **2 variables les plus discriminantes** en explorant les corrélations
2. Implémenter la régression logistique **à la main** sur ces 2 variables → visualiser la frontière de décision
3. Entraîner Régression Logistique et k-NN sklearn sur **les 30 variables** → comparer les performances
4. Discuter **précision vs rappel** — pourquoi l'exactitude seule est trompeuse en contexte médical


## Étape 1 — Charger le jeu de données et explorer l'équilibre des classes

In [ ]:
# Load the Breast Cancer Wisconsin dataset — given
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df_bc = pd.DataFrame(data.data, columns=data.feature_names)
df_bc['target'] = data.target

print("Dimensions :", df_bc.shape)
print("\nDistribution des classes :")
print(df_bc['target'].value_counts().rename({0: "malin", 1: "bénin"}))

## Étape 2 — Trouver les 2 variables les plus discriminantes

In [ ]:
#TODO : calculer la corrélation de Pearson absolue de chaque variable avec la colonne cible
# Indice : df_bc.drop(columns='target').corrwith(df_bc['target']).abs()
# Puis trier par ordre décroissant
correlations = ...

print("Top 10 variables par corrélation avec la cible :")
print(correlations.head(10).round(3))

In [ ]:
# Diagramme à barres des 15 premières corrélations — fourni
plt.figure(figsize=(10, 4))
correlations.head(15).plot(kind='bar', color='steelblue', edgecolor='k')
plt.title("Corrélation des variables avec la cible (malin vs bénin)")
plt.ylabel("Corrélation absolue")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#TODO : extraire les noms des 2 meilleures variables et construire X2, y2
# Indice : correlations.head(2).index.tolist()
top2 = ...

X2 = ...   # numpy array, shape (N, 2)
y2 = df_bc['target'].values

print("Variables sélectionnées :", top2)

## Étape 3 — Visualiser l'espace à 2 variables

In [ ]:
# Nuage de points — fourni
plt.figure(figsize=(7, 5))
for label, color, name in zip([0, 1], ['red', 'steelblue'], ['malignant', 'benign']):
    mask = y2 == label
    plt.scatter(X2[mask, 0], X2[mask, 1],
                c=color, label=name, alpha=0.5, edgecolors='k', s=30)
plt.xlabel(top2[0])
plt.ylabel(top2[1])
plt.title("Cancer du Sein — 2 variables les plus discriminantes")
plt.legend()
plt.tight_layout()
plt.show()

### 📝 Observations

*Les deux classes semblent-elles linéairement séparables dans cet espace 2D ?  
Pensez-vous qu'une frontière logistique fonctionnera bien ici ?*



## Étape 4 — Régression Logistique à la Main sur 2 Variables

Nous réutilisons `sigmoid`, `binary_cross_entropy` et `gradient_descent` de la Partie 1.


In [ ]:
#TODO : normaliser X2 avec StandardScaler et diviser en train/test
scaler2 = StandardScaler()
X2_scaled = ...

X2_train, X2_test, y2_train, y2_test = train_test_split(
    ..., ..., test_size=0.2, random_state=0
)

In [ ]:
#TODO : lancer gradient_descent sur les données cancer à 2 variables
# Utiliser par exemple lr=0.1 et epochs=500
w2, losses2 = ...

# Training loss plot — given
plt.figure(figsize=(6, 3))
plt.plot(losses2, color='steelblue')
plt.title("Perte d'entraînement — régression logistique à la main (2 variables)")
plt.xlabel("Époque")
plt.ylabel("Perte")
plt.tight_layout()
plt.show()

In [ ]:
#TODO : calculer les probabilités prédites sur X2_test avec sigmoid
# Puis seuiller à 0.5 pour obtenir des prédictions binaires
# Enfin calculer et afficher la précision
p2_test = ...
y2_pred = ...
acc2    = ...

print(f"Précision régression logistique à la main (2 variables) : {acc2:.3f}")

## Étape 5 — Tracer la frontière de décision (2 variables)

In [ ]:
# Tracé de la frontière de décision — fourni
# La frontière vérifie w0*x0 + w1*x1 = 0  →  x1 = -(w0/w1)*x0
x0_range = np.linspace(X2_scaled[:, 0].min() - 0.5,
                        X2_scaled[:, 0].max() + 0.5, 200)
x1_boundary = -(w2[0] / w2[1]) * x0_range

plt.figure(figsize=(7, 5))
for label, color, name in zip([0, 1], ['red', 'steelblue'], ['malignant', 'benign']):
    mask = y2 == label
    plt.scatter(X2_scaled[mask, 0], X2_scaled[mask, 1],
                c=color, label=name, alpha=0.5, edgecolors='k', s=30)
plt.plot(x0_range, x1_boundary, 'k--', lw=2, label='Frontière de décision')
plt.xlabel(top2[0] + " (scaled)")
plt.ylabel(top2[1] + " (scaled)")
plt.title("Régression logistique à la main — frontière de décision")
plt.legend()
plt.tight_layout()
plt.show()

## Étape 6 — Modèles Complets avec les 30 Variables (sklearn)

Nous utilisons maintenant les 30 variables et comparons Régression Logistique et k-NN.


In [ ]:
# Préparer le jeu complet — fourni
X_full = data.data
y_full = data.target

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full, y_full, test_size=0.2, random_state=0
)

scaler_f = StandardScaler()
X_train_f = scaler_f.fit_transform(X_train_f)
X_test_f  = scaler_f.transform(X_test_f)

In [ ]:
#TODO : entraîner LogisticRegression et KNeighborsClassifier sur les 30 variables
# Afficher la précision des deux modèles
log_full = ...
knn_full = ...

# Entraîner les deux modèles
...

for name, model in [("Régression Logistique", log_full), ("k-NN (k=5)", knn_full)]:
    acc = ...
    print(f"{name:25s} | Précision : {acc:.3f}")

In [ ]:
# Matrices de confusion côte à côte — fournies
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model, name in zip(
    axes,
    [log_full, knn_full],
    ["Régression Logistique", "k-NN (k=5)"]
):
    cm = confusion_matrix(y_test_f, model.predict(X_test_f))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['malin', 'bénin'],
                yticklabels=['malin', 'bénin'])
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
    ax.set_title(name)

plt.tight_layout()
plt.show()

## Étape 7 — Précision vs Rappel : Pourquoi l'Exactitude ne Suffit Pas

Dans un contexte médical, les deux types d'erreurs ne sont **pas équivalents** :

| Erreur | Signification | Conséquence |
|---|---|---|
| **Faux Négatif** | Prédit bénin, en réalité malin | ⚠️ Patient non traité |
| **Faux Positif** | Prédit malin, en réalité bénin | Examens supplémentaires, stress |

Un **faux négatif est bien plus dangereux** ici.
Cela signifie qu'on doit privilégier le **rappel** (sensibilité) par rapport à la précision pour la classe maligne.

$$\text{Recall} = \frac{TP}{TP + FN}$$

$$\text{Precision} = \frac{TP}{TP + FP}$$


In [ ]:
#TODO : afficher le classification_report complet pour les deux modèles
# Noms des classes : ['malin (0)', 'bénin (1)']
# Attention particulière au rappel pour la classe 0
from sklearn.metrics import classification_report

for name, model in [("Régression Logistique", log_full), ("k-NN", knn_full)]:
    print(f"=== {name} ===")
    # Votre code ici


### 📝 Observations

*Quel modèle a le meilleur rappel pour la classe maligne ?  
Y a-t-il un compromis précision/rappel à discuter ?  
Quel seuil choisiriez-vous dans un vrai contexte clinique ?*



## Étape 8 — Courbe ROC et Score AUC

La **courbe ROC** trace le Taux de Vrais Positifs (rappel) en fonction du Taux de Faux Positifs selon le seuil de décision.

$AUC = 1$ est parfait, $AUC = 0.5$ est aléatoire.


In [ ]:
# Courbes ROC — fournies
from sklearn.metrics import roc_curve, roc_auc_score

plt.figure(figsize=(7, 5))

for model, name, color in [
    (log_full, "Régression Logistique", "steelblue"),
    (knn_full, "k-NN (k=5)",          "tomato")
]:
    proba = model.predict_proba(X_test_f)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_f, proba)
    auc = roc_auc_score(y_test_f, proba)
    plt.plot(fpr, tpr, lw=2, color=color, label=f"{name}  (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Classifieur aléatoire')
plt.xlabel("Taux de Faux Positifs")
plt.ylabel("Taux de Vrais Positifs (Rappel)")
plt.title("Courbe ROC — Cancer du Sein")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#TODO : expérimenter avec un seuil de décision plus bas (ex. 0.3) pour la régression logistique
# Le rappel pour les malins augmente-t-il ? Quel est le coût ?
threshold = 0.3

proba_log = log_full.predict_proba(X_test_f)[:, 1]

#TODO : appliquer le seuil pour obtenir les prédictions binaires y_pred_adjusted
y_pred_adjusted = ...

print(f"=== Régression Logistique avec seuil = {threshold} ===")
print(classification_report(y_test_f, y_pred_adjusted,
                             target_names=['malin (0)', 'bénin (1)']))

### 📝 Observations

*Qu'arrive-t-il à la précision et au rappel quand vous baissez le seuil ?  
Comment choisiriez-vous le seuil en pratique ?*



## Résumé — Points Clés

| Étape | Point clé |
|---|---|
| Modèle à 2 variables | Une frontière linéaire est visible et interprétable |
| Modèle complet 30 variables | Les performances augmentent significativement |
| Logistique vs k-NN | Les deux sont bons ; la régression logistique est plus interprétable |
| Précision vs Rappel | En médecine, **le rappel sur la classe dangereuse prime** |
| ROC / AUC | Meilleure évaluation que l'exactitude pour les tâches déséquilibrées ou à enjeux |
| Ajustement du seuil | Baisser le seuil augmente le rappel au détriment de la précision |
